# MolScope Matched Benchmark: HP35 Villin Headpiece (2RJY)

This interactive notebook executes and visualizes MolScope's full analytical continuum on the canonical fast-folding **HP35 Villin Headpiece** (PDB: `2RJY`), profiling:
- **Computational Performance**: Wall-clock duration across feature extraction, TICA, Contact KPCA, DPA clustering, and Decision Tree explainability.
- **Memory Efficiency**: Peak RSS memory tracking and heap delta.
- **Biophysical State Recovery**: Metastable state identification and Free Energy Landscape profiling.
- **Explainable AI**: Boolean structural rule extraction and 3D PyMOL / ChimeraX mapping.


## 1. Imports and System Initialization

We initialize the benchmarking environment, import `molscope.api.Study`, and prepare performance profilers.

In [ ]:
import os
import sys
import time
import json
from pathlib import Path
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from molscope.api import Study

out_dir = Path("results/2rjy_benchmark")
out_dir.mkdir(parents=True, exist_ok=True)
process = psutil.Process()
start_mem = process.memory_info().rss / (1024 * 1024)
print(f"Initial process RSS memory: {start_mem:.2f} MB")


## 2. Ingesting System Data into Study

We configure the 2RJY system. If the full 10,001-frame trajectory (`protein.pdb` and `simulation_run1.xtc`) is present in `data/2RJY/`, it runs on real simulation data; otherwise, it automatically falls back to bundled fixtures for immediate verification.

In [ ]:
study = Study(
    project_id="2rjy_matched_benchmark",
    out_dir=str(out_dir),
)

data_candidates = [
    Path("data/2RJY"),
    Path("../data/2RJY"),
    Path("../../data/2RJY"),
    Path("tests/fixtures"),
    Path("../tests/fixtures"),
]

loaded_system = False
for base in data_candidates:
    top = base / "protein.pdb"
    traj = base / "simulation_run1.xtc"
    if top.is_file() and traj.is_file():
        study.add_system(
            system_id="villin_headpiece",
            topology=str(top.resolve()),
            trajectories=[str(traj.resolve())],
        )
        print(f"Loaded full 2RJY benchmark inputs from: {base}")
        loaded_system = True
        break

if not loaded_system:
    # Fallback to test fixtures for demonstration
    fixture_top = next((b / "1A22.pdb" for b in data_candidates if (b / "1A22.pdb").is_file()), None)
    fixture_traj = next((b / "1A22_traj.dcd" for b in data_candidates if (b / "1A22_traj.dcd").is_file()), None)
    if fixture_top and fixture_traj:
        study.add_system(
            system_id="demo_benchmark",
            topology=str(fixture_top.resolve()),
            trajectories=[str(fixture_traj.resolve())],
        )
        print(f"Loaded benchmark fixture demo from: {fixture_top.parent}")
    else:
        print("No trajectory files found. Place protein.pdb and simulation_run1.xtc in data/2RJY/")


## 3. Benchmark Step 1: Feature Extraction

We configure on-the-fly PBC imaging and 3 biophysical feature families (Cα contacts, DSSP secondary structure fractions, sidechain rotamers) and measure extraction time.

In [ ]:
study.configure_imaging(unwrap=True, center_selection="protein", wrap=False)
study.configure_features([
    "protein.ca_distances_contacts",
    "protein.secondary_structure_dssp",
    "protein.sidechain_torsions",
])
study.set_plugin_parameters("protein.ca_distances_contacts", {
    "selection": "protein and name CA",
    "contact_cutoff_angstrom": 8.0,
    "max_pairs": 3000,
})
study.set_plugin_parameters("protein.sidechain_torsions", {
    "residue_ids": [17, 20, 23, 27, 30, 35, 40, 50],
})

t0_extract = time.perf_counter()
study.run(extract_only=True)
t_extract = time.perf_counter() - t0_extract
features = study.get_features()
manifest = study.get_manifest()
print(f"Feature Extraction Complete: {features.shape[1]} features across {features.shape[0]} frames in {t_extract:.2f}s")


## 4. Benchmark Step 2: Conformational Dimensionality Reduction (TICA & CKPCA)

We fit both Variational TICA (to identify the slowest kinetic modes) and Contact Kernel PCA (CKPCA, to capture non-linear contact manifolds).

In [ ]:
# Fit TICA (lag time = 2 frames)
t0_tica = time.perf_counter()
tica_res = study.run_tica(lag_time=2, components=5)
t_tica = time.perf_counter() - t0_tica
print(f"TICA: Implied timescales = {[round(t, 1) for t in tica_res.implied_timescales[:3]]} ps (fitted in {t_tica:.2f}s)")

# Fit Contact KPCA
t0_ckpca = time.perf_counter()
ckpca_res = study.run_contact_kpca(n_components=5)
t_ckpca = time.perf_counter() - t0_ckpca
print(f"CKPCA: Explained variance = {[round(v, 4) for v in ckpca_res.explained_variance_ratio[:3]]} (fitted in {t_ckpca:.2f}s)")


## 5. Benchmark Step 3: Density Peak Advanced (DPA) Clustering & 2D FES

We run non-parametric DPA clustering in the kinetic reaction space and compute the 2D Free Energy Surface $\Delta G(x, y)$.

In [ ]:
# DPA clustering
t0_dpa = time.perf_counter()
dpa_res = study.cluster_dpa(coordinate_name="TICA", z_score=2.0)
t_dpa = time.perf_counter() - t0_dpa
print(f"DPA: Discovered {dpa_res.n_clusters} conformational basins in {t_dpa:.2f}s")

# Compute 2D Free Energy Surface
fes = study.plot_free_energy_landscape(
    dim1="TIC1",
    dim2="TIC2",
    temperature_kelvin=300.0,
    energy_units="kcal/mol",
    out_path=str(out_dir / "fes_tic1_tic2.png"),
    interactive_html=str(out_dir / "fes_tic1_tic2.html"),
)
print(f"Detected {len(fes.basins)} free energy minima.")


## 6. Benchmark Step 4: Explainability & 3D Export

We fit regularized Decision Trees to extract human-readable Boolean if-then rules explaining state transitions, and export 3D PyMOL / ChimeraX scripts.

In [ ]:
t0_dt = time.perf_counter()
explainer = study.explain_clusters(clustering_name="dpa", max_depth=3)
t_dt = time.perf_counter() - t0_dt
print(f"Decision Tree Explainability complete in {t_dt:.2f}s")

for r in explainer.rules[:3]:
    print(f"  State {r.cluster_id}: {r.simplified_rule_text()}")

# Export 3D visualization
viz_files = study.export_3d(out_dir=str(out_dir / "3d_visualization"))
print(f"Exported 3D visualizers: {list(viz_files.keys())}")


## 7. Performance Diagnostics & Scaling Visualization

We analyze the recorded wall-clock durations and memory footprint across the entire benchmark continuum.

In [ ]:
peak_mem = process.memory_info().rss / (1024 * 1024)
mem_delta = peak_mem - start_mem
total_time = t_extract + t_tica + t_ckpca + t_dpa + t_dt

stages = ["Feature Extraction", "TICA", "Contact KPCA", "DPA Clustering", "Decision Trees"]
times = [t_extract, t_tica, t_ckpca, t_dpa, t_dt]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Runtime breakdown
bars = ax1.barh(stages, times, color="#3182ce", edgecolor="black", alpha=0.85)
ax1.set_xlabel("Wall-Clock Time (seconds)")
ax1.set_title(f"Stage Duration Breakdown (Total: {total_time:.2f}s)")
ax1.grid(True, linestyle="--", alpha=0.4)
for bar in bars:
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.2f}s", va="center", fontsize=10)

# State population distribution
state_labels = [f"State {k}" for k in dpa_res.state_populations.keys()]
pops = list(dpa_res.state_populations.values())
ax2.pie(pops, labels=state_labels, autopct="%1.1f%%", colors=plt.cm.tab10.colors[:len(pops)], startangle=140)
ax2.set_title(f"Conformational State Populations ({len(pops)} States)")

plt.tight_layout()
plt.show()

print(f"\nSummary Performance Metrics:")
print(f"  Total Pipeline Time : {total_time:.2f} seconds")
print(f"  Peak Process Memory : {peak_mem:.1f} MB (Heap Delta: +{mem_delta:.1f} MB)")
print(f"  Analyzed Frames     : {len(features):,}")
print(f"  Extracted Features  : {features.shape[1]:,}")
